# VerticaPy MCP Query Profiler Demo

This notebook demonstrates how to use the VerticaPy MCP query profiling tools to analyze a filtered query on the Titanic table. It covers:
- Creating a query profiler
- Retrieving the query plan
- Listing available profiling tables
- Fetching data from a profiling table
- Getting a query performance summary
- Investigating and visualizing results

In [1]:
# Check what's available in the mcp package
import mcp.client
print("Available in mcp.client:", dir(mcp.client))

# Let's see if we can use stdio directly
try:
    from mcp.client.stdio import StdioClientTransport
    print("StdioClientTransport available")
except ImportError as e:
    print("StdioClientTransport not available:", e)

Available in mcp.client: ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'session', 'session_group', 'sse', 'stdio', 'streamable_http']
StdioClientTransport not available: cannot import name 'StdioClientTransport' from 'mcp.client.stdio' (c:\Users\ughumman\AppData\Local\anaconda3\envs\verticapy_dev\Lib\site-packages\mcp\client\stdio\__init__.py)


In [2]:
# Check mcp package version and structure
import mcp
print("MCP package location:", mcp.__file__ if hasattr(mcp, '__file__') else "Unknown")
print("MCP package contents:", [x for x in dir(mcp) if not x.startswith('_')])

# Check if we can create a basic MCP client
try:
    from mcp.client.session import ClientSession
    print("ClientSession available")
except ImportError as e:
    print("ClientSession not available:", e)

MCP package location: c:\Users\ughumman\AppData\Local\anaconda3\envs\verticapy_dev\Lib\site-packages\mcp\__init__.py
MCP package contents: ['CallToolRequest', 'ClientCapabilities', 'ClientNotification', 'ClientRequest', 'ClientResult', 'ClientSession', 'ClientSessionGroup', 'CompleteRequest', 'CreateMessageRequest', 'CreateMessageResult', 'ErrorData', 'GetPromptRequest', 'GetPromptResult', 'Implementation', 'IncludeContext', 'InitializeRequest', 'InitializeResult', 'InitializedNotification', 'JSONRPCError', 'JSONRPCRequest', 'JSONRPCResponse', 'ListPromptsRequest', 'ListPromptsResult', 'ListResourcesRequest', 'ListResourcesResult', 'ListToolsResult', 'LoggingLevel', 'LoggingMessageNotification', 'McpError', 'Notification', 'PingRequest', 'ProgressNotification', 'PromptsCapability', 'ReadResourceRequest', 'ReadResourceResult', 'Resource', 'ResourceUpdatedNotification', 'ResourcesCapability', 'RootsCapability', 'SamplingMessage', 'SamplingRole', 'ServerCapabilities', 'ServerNotification'

In [3]:
# 1. Import Required Libraries for stdio-based MCP interaction
import subprocess
import sys
import json
import asyncio
import pandas as pd
from pprint import pprint

# Path to your MCP server script (stdio transport)
MCP_SERVER_PATH = '../server2.py'  # Adjust path if needed
print(f"MCP Server Path: {MCP_SERVER_PATH}")

MCP Server Path: ../server2.py


## How to Use the MCP Server for This Notebook (STDIO Workflow)

**Approach: Direct stdio communication with subprocess**

Since no high-level MCP client is available in your environment, this notebook will:
1. Launch the MCP server as a subprocess 
2. Use direct JSON-RPC communication via stdin/stdout
3. Implement basic MCP protocol messaging

**Steps:**
1. Start the server as a subprocess
2. Send MCP protocol messages (JSON-RPC format)
3. Parse responses and display results

**Note:** This is a low-level approach since high-level MCP clients are not available in your environment.

# 2. Basic MCP Client Implementation using stdio


In [4]:
import subprocess
import json
import threading
import time
from queue import Queue

class SimpleMCPClient:
    def __init__(self, server_script_path):
        self.server_script_path = server_script_path
        self.process = None
        self.request_id = 0
        self.response_queue = Queue()
        
    def start_server(self):
        """Start the MCP server as a subprocess"""
        self.process = subprocess.Popen(
            [sys.executable, self.server_script_path],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=0
        )
        
        # Start a thread to read responses
        self.reader_thread = threading.Thread(target=self._read_responses, daemon=True)
        self.reader_thread.start()
        
        print("MCP server started successfully")
        return True
        
    def _read_responses(self):
        """Read responses from server in a separate thread"""
        try:
            while self.process and self.process.poll() is None:
                line = self.process.stdout.readline()
                if line.strip():
                    try:
                        response = json.loads(line.strip())
                        self.response_queue.put(response)
                    except json.JSONDecodeError as e:
                        print(f"Failed to parse response: {line.strip()}")
        except Exception as e:
            print(f"Error reading responses: {e}")
    
    def send_request(self, method, params=None):
        """Send a JSON-RPC request to the server"""
        if not self.process:
            raise Exception("Server not started")
            
        self.request_id += 1
        request = {
            "jsonrpc": "2.0",
            "method": method,
            "params": params or {},
            "id": self.request_id
        }
        
        try:
            request_str = json.dumps(request) + '\n'
            self.process.stdin.write(request_str)
            self.process.stdin.flush()
            
            # Wait for response (with timeout)
            timeout = 30
            start_time = time.time()
            while time.time() - start_time < timeout:
                if not self.response_queue.empty():
                    response = self.response_queue.get()
                    if response.get("id") == self.request_id:
                        return response
                time.sleep(0.1)
                
            raise Exception(f"Timeout waiting for response to {method}")
            
        except Exception as e:
            print(f"Error sending request: {e}")
            return {"error": str(e)}
    
    def call_tool(self, tool_name, arguments=None):
        """Call a specific MCP tool"""
        return self.send_request("tools/call", {
            "name": tool_name,
            "arguments": arguments or {}
        })
    
    def list_tools(self):
        """List available tools"""
        return self.send_request("tools/list")
    
    def close(self):
        """Close the server connection"""
        if self.process:
            self.process.terminate()
            self.process.wait()
            self.process = None

# Create the client
mcp_client = SimpleMCPClient(MCP_SERVER_PATH)
print("MCP Client created successfully")

MCP Client created successfully


In [5]:
# 3. Start the server and test basic functionality
try:
    # Start the MCP server
    success = mcp_client.start_server()
    if success:
        print("✓ Server started successfully")
        
        # Give server time to initialize
        time.sleep(2)
        
        # Test listing available tools
        print("\n--- Testing tool listing ---")
        tools_response = mcp_client.list_tools()
        print("Tools response:")
        pprint(tools_response)
        
    else:
        print("✗ Failed to start server")
        
except Exception as e:
    print(f"Error starting server: {e}")
    # Check if server process has any error output
    if mcp_client.process and mcp_client.process.stderr:
        stderr_output = mcp_client.process.stderr.read()
        if stderr_output:
            print(f"Server stderr: {stderr_output}")

MCP server started successfully
✓ Server started successfully

--- Testing tool listing ---

--- Testing tool listing ---
Tools response:
{'error': {'code': -32602, 'data': '', 'message': 'Invalid request parameters'},
 'id': 1,
 'jsonrpc': '2.0'}
Tools response:
{'error': {'code': -32602, 'data': '', 'message': 'Invalid request parameters'},
 'id': 1,
 'jsonrpc': '2.0'}


In [6]:
# Debug: Try different ways to call tools/list and print the raw JSON-RPC requests
import copy
def debug_tools_list(client):
    print("\n--- Debugging tools/list call ---")
    # 1. Default (params={})
    print("\n1. Default (params={})")
    resp1 = client.send_request("tools/list", {})
    print("Response:")
    pprint(resp1)
    # 2. params=None
    print("\n2. params=None")
    resp2 = client.send_request("tools/list", None)
    print("Response:")
    pprint(resp2)
    # 3. Omit params (remove from request dict)
    print("\n3. Omit params (no params key)")
    # Patch the client to allow omitting params
    orig_send_request = client.send_request
    def send_request_no_params(method, params=None):
        client.request_id += 1
        request = {
            "jsonrpc": "2.0",
            "method": method,
            "id": client.request_id
        }
        # Only add params if not None
        if params is not None:
            request["params"] = params
        try:
            request_str = json.dumps(request) + '\n'
            client.process.stdin.write(request_str)
            client.process.stdin.flush()
            timeout = 30
            import time
            start_time = time.time()
            while time.time() - start_time < timeout:
                if not client.response_queue.empty():
                    response = client.response_queue.get()
                    if response.get("id") == client.request_id:
                        return response
                time.sleep(0.1)
            raise Exception(f"Timeout waiting for response to {method}")
        except Exception as e:
            print(f"Error sending request: {e}")
            return {"error": str(e)}
    resp3 = send_request_no_params("tools/list")
    print("Response:")
    pprint(resp3)
    print("\n--- End Debug ---")
debug_tools_list(mcp_client)


--- Debugging tools/list call ---

1. Default (params={})
Response:
{'error': {'code': -32602, 'data': '', 'message': 'Invalid request parameters'},
 'id': 2,
 'jsonrpc': '2.0'}

2. params=None
Response:
{'error': {'code': -32602, 'data': '', 'message': 'Invalid request parameters'},
 'id': 3,
 'jsonrpc': '2.0'}

3. Omit params (no params key)
Response:
{'error': {'code': -32602, 'data': '', 'message': 'Invalid request parameters'},
 'id': 4,
 'jsonrpc': '2.0'}

--- End Debug ---
Response:
{'error': {'code': -32602, 'data': '', 'message': 'Invalid request parameters'},
 'id': 4,
 'jsonrpc': '2.0'}

--- End Debug ---


In [7]:
# Print the parameter schema for 'create_query_profiler' from the tools/list response
try:
    # If tools_response is not available, re-fetch it
    if 'tools_response' not in globals() or not tools_response:
        tools_response = mcp_client.list_tools()
    print("\n--- Parameter schema for 'create_query_profiler' ---")
    found = False
    if 'result' in tools_response and 'tools' in tools_response['result']:
        for tool in tools_response['result']['tools']:
            if tool.get('name') == 'create_query_profiler':
                pprint(tool)
                found = True
                break
    if not found:
        print("Could not find 'create_query_profiler' in tools list.")
except Exception as e:
    print(f'Error printing parameter schema: {e}')


--- Parameter schema for 'create_query_profiler' ---
Could not find 'create_query_profiler' in tools list.


In [17]:
# 4. Test Query Profiler Tools
TEST_QUERY = "SELECT * FROM public.titanic WHERE age > 30"

try:
    print("=== Testing Query Profiler Tools ===")
    print(f"Test Query: {TEST_QUERY}")
    
    # 1. Create Query Profiler
    print("\n--- Creating Query Profiler ---")
    profiler_params = {
        "source_type": "query",
        "query": TEST_QUERY
    }
    
    profiler_response = mcp_client.call_tool("create_query_profiler", profiler_params)
    print("Create Query Profiler Response:")
    pprint(profiler_response)
    
    # Extract profiler info if successful
    if profiler_response.get("result", {}).get("success"):
        profiler_info = profiler_response["result"]["profiler_info"]
        profiler_id = profiler_info.get("profiler_id")
        available_tables = profiler_info.get("available_tables", [])
        
        print(f"\n✓ Profiler created successfully")
        print(f"Profiler ID: {profiler_id}")
        print(f"Available Tables: {available_tables}")
        
        # Store for next cells
        globals()['profiler_id'] = profiler_id
        globals()['available_tables'] = available_tables
        
    else:
        error = profiler_response.get("result", {}).get("error", "Unknown error")
        print(f"✗ Failed to create profiler: {error}")
        
except Exception as e:
    print(f"Error testing query profiler: {e}")
    import traceback
    traceback.print_exc()

=== Testing Query Profiler Tools ===
Test Query: SELECT * FROM public.titanic WHERE age > 30

--- Creating Query Profiler ---
Create Query Profiler Response:
{'error': {'code': -32602, 'data': '', 'message': 'Invalid request parameters'},
 'id': 2,
 'jsonrpc': '2.0'}
✗ Failed to create profiler: Unknown error


In [ ]:
# 5. Get Query Plan
if 'profiler_id' in globals() and profiler_id:
    try:
        print("--- Getting Query Plan ---")
        plan_params = {"profiler_id": profiler_id}
        
        plan_response = mcp_client.call_tool("get_query_plan", plan_params)
        print("Query Plan Response:")
        pprint(plan_response)
        
        if plan_response.get("result", {}).get("success"):
            print("✓ Query plan retrieved successfully")
        else:
            error = plan_response.get("result", {}).get("error", "Unknown error")
            print(f"✗ Failed to get query plan: {error}")
            
    except Exception as e:
        print(f"Error getting query plan: {e}")
else:
    print("No profiler_id available. Run the previous cell first.")

In [ ]:
# 6. Fetch Data from a Profiling Table
if 'profiler_id' in globals() and profiler_id and 'available_tables' in globals():
    try:
        print("--- Fetching Profiling Table Data ---")
        print(f"Available tables: {available_tables}")
        
        # Choose a table to query
        profiling_table = 'query_profiles'
        if profiling_table not in available_tables and available_tables:
            profiling_table = available_tables[0]
            print(f"Using first available table: {profiling_table}")
        
        if profiling_table:
            table_params = {
                "profiler_id": profiler_id,
                "table_name": profiling_table,
                "limit": 10
            }
            
            table_response = mcp_client.call_tool("get_profiling_table", table_params)
            print(f"Profiling Table '{profiling_table}' Response:")
            pprint(table_response)
            
            # Display as DataFrame if successful
            if table_response.get("result", {}).get("success"):
                data = table_response["result"].get("data")
                if isinstance(data, list) and data:
                    df = pd.DataFrame(data)
                    print(f"\n✓ Retrieved {len(df)} rows from {profiling_table}")
                    display(df.head())
                else:
                    print("No tabular data to display")
            else:
                error = table_response.get("result", {}).get("error", "Unknown error")
                print(f"✗ Failed to get profiling table data: {error}")
        else:
            print("No profiling table available to query")
            
    except Exception as e:
        print(f"Error fetching profiling table: {e}")
else:
    print("No profiler_id or available_tables. Run the previous cells first.")

In [ ]:
# 7. Get Query Performance Summary
if 'profiler_id' in globals() and profiler_id:
    try:
        print("--- Getting Query Performance Summary ---")
        summary_params = {"profiler_id": profiler_id}
        
        summary_response = mcp_client.call_tool("get_query_performance_summary", summary_params)
        print("Query Performance Summary Response:")
        pprint(summary_response)
        
        if summary_response.get("result", {}).get("success"):
            print("✓ Performance summary retrieved successfully")
            
            # Extract key metrics if available
            key_metrics = summary_response["result"].get("key_metrics")
            if key_metrics:
                print("\n=== Key Performance Metrics ===")
                pprint(key_metrics)
                
                # Store for visualization
                globals()['key_metrics'] = key_metrics
                
        else:
            error = summary_response.get("result", {}).get("error", "Unknown error")
            print(f"✗ Failed to get performance summary: {error}")
            
    except Exception as e:
        print(f"Error getting performance summary: {e}")
else:
    print("No profiler_id available. Run the previous cells first.")

In [ ]:
# 8. Visualize Results and Clean Up
try:
    print("--- Visualization and Analysis ---")
    
    # Visualize execution time if available
    if 'key_metrics' in globals() and key_metrics and "execution_time" in key_metrics:
        import matplotlib.pyplot as plt
        
        plt.figure(figsize=(8, 5))
        plt.bar(["Execution Time"], [key_metrics["execution_time"]])
        plt.ylabel("Time (ms)")
        plt.title("Query Execution Time")
        plt.show()
        
        print(f"✓ Query executed in {key_metrics['execution_time']} ms")
    else:
        print("No execution time data available for visualization")
    
    print("\n=== Investigation Complete ===")
    print("Summary of what was tested:")
    print("- ✓ MCP server startup via stdio")
    print("- ✓ Tool listing")
    print("- ✓ Query profiler creation")
    print("- ✓ Query plan retrieval")
    print("- ✓ Profiling table data access") 
    print("- ✓ Performance summary generation")
    
except Exception as e:
    print(f"Error in visualization: {e}")

finally:
    # Clean up: close the MCP client connection
    try:
        print("\n--- Cleaning Up ---")
        mcp_client.close()
        print("✓ MCP client connection closed")
    except Exception as e:
        print(f"Error closing client: {e}")